In [37]:
import os
import time
import base64
import json
import pandas as pd
from tqdm import tqdm
from mistralai.client import Mistral
import fitz 
from huggingface_hub import InferenceClient

In [ ]:
from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

In [ ]:
BASE_PATH = "../"
IMAGE_SAVE_DIR = os.path.join(BASE_PATH, "data", "images") 

In [ ]:
import os
import re
import time
import base64
import fitz
import pandas as pd
from tqdm import tqdm
from mistralai.client import Mistral

def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def parse_pdf_to_slides_with_images(pdf_path, output_image_dir):
    os.makedirs(output_image_dir, exist_ok=True)
    slides = []
    doc = fitz.open(pdf_path)
    base_name = os.path.basename(pdf_path).replace('.pdf', '')
    
    for i, page in enumerate(doc):
        text = page.get_text("text").strip()
        clean_text = re.sub(r' {2,}', ' ', text)
        
        pix = page.get_pixmap(dpi=150)
        img_path = os.path.join(output_image_dir, f"{base_name}_slide_{i+1}.png")
        pix.save(img_path)
        
        slides.append({
            "id": f"{base_name}_slide_{i+1}", 
            "content": clean_text,
            "image_path": img_path
        })
            
    doc.close()
    return slides

In [40]:
def get_slide_description_prompt(text_content):
    return f"""### РОЛЬ
Ты -- лектор в университете на кафедре Computer Science и Machine Learning. 
Твоя задача: посмотреть на слайд (и его извлеченный текст) и произнести его вслух так, как это сделал бы живой человек на лекции.

### ИЗВЛЕЧЕННЫЙ ТЕКСТ СЛАЙДА (может быть с ошибками или неполным):
{text_content}

### ИНСТРУКЦИИ
1. Сделай описание материала со слайда, опираясь НА КАРТИНКУ (графики, схемы).
2. Если на слайде много информации, в речи оставь произвольную часть, но НЕ МЕНЕЕ половины информации слайда должно быть покрыто.
3. Описывай код максимально подробно, каждую часть алгоритма.
4. Ответ должен состоять из НЕ МЕНЕЕ 6 ПРЕДЛОЖЕНИЙ.

### ОГРАНИЧЕНИЯ (Negative Constraints)
- НЕ добавляй вводных фраз.
- НЕ добавляй примечаний от редактора.
- НЕ меняй смысл высказывания.
- НЕ приводи отвлеченных примеров.
- НЕ заключай ответ в кавычки.
"""

In [50]:
def generate_code_description(client, text_content, num_samples=3, max_retries=3):
    """Fallback generator for code chunks (which have no images)"""
    prompt = f"""### РОЛЬ
Ты -- лектор в университете на кафедре Computer Science и Machine Learning. 
Твоя задача: посмотреть на слайд (и его извлеченный текст) и произнести его вслух так, как это сделал бы живой человек на лекции.

### ИЗВЛЕЧЕННЫЙ ТЕКСТ СЛАЙДА (может быть с ошибками или неполным):
{text_content}

### ИНСТРУКЦИИ
1. Сделай описание материала со слайда, опираясь НА КАРТИНКУ (графики, схемы).
2. Если на слайде много информации, в речи оставь произвольную часть, но НЕ МЕНЕЕ половины информации слайда должно быть покрыто.
3. Описывай код максимально подробно, каждую часть алгоритма.
4. Ответ должен состоять из НЕ МЕНЕЕ 6 ПРЕДЛОЖЕНИЙ.

### ОГРАНИЧЕНИЯ (Negative Constraints)
- НЕ добавляй вводных фраз.
- НЕ добавляй примечаний от редактора.
- НЕ меняй смысл высказывания.
- НЕ приводи отвлеченных примеров.
- НЕ заключай ответ в кавычки.
"""
    result = []
    for _ in range(num_samples):
        for attempt in range(max_retries):
            try:
                response = client.chat.complete(
                    model="mistral-small-latest",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.7 
                )
                result.append(response.choices[0].message.content.strip())
            except Exception as e:
                print(f"API Error (Code): {e}")
                time.sleep(2 ** attempt)
    return result

In [73]:
def generate_slide_description_mistral(client: Mistral, text_content: str, image_path: str, num_samples=3, max_retries=5):
    base64_image = encode_image_to_base64(image_path)
    prompt = get_slide_description_prompt(text_content)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": f"data:image/jpeg;base64,{base64_image}"}
            ]
        }
    ]
    
    result = []
    for _ in range(num_samples):
        for attempt in range(max_retries):
            try:
                response = client.chat.complete(
                    # model="pixtral-12b-2409", 
                    model="mistral-small-2603", 
                    messages=messages,
                    temperature=0.7 
                )
                result.append(response.choices[0].message.content.strip())
            except Exception as e:
                print(f"API Error: {e}")
                if attempt == max_retries - 1:
                    print(f"Failed to generate descriptions for {image_path}")
                time.sleep(4 + 2 ** attempt)
    return result

In [74]:
def generate_slide_description_hf(client: InferenceClient, text_content: str, image_path: str, num_samples=3, max_retries=3):
    base64_image = encode_image_to_base64(image_path)
    prompt = get_slide_description_prompt(text_content)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        }
    ]
    
    result = []
    for _ in range(num_samples):
        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model="meta-llama/Llama-3.2-11B-Vision-Instruct", 
                    messages=messages, 
                    max_tokens=500
                )
                result.append(response.choices[0].message.content.strip())
            except Exception as e:
                print(f"HF API Error: {e}")
                time.sleep(2 ** attempt)
    return result

In [75]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

def parse_code_to_blocks(file_path):
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    ext = file_path.split('.')[-1].lower()
    
    lang_map = {
        'py': Language.PYTHON,
        'cpp': Language.CPP,
        'c': Language.C,
        'h': Language.CPP,
        'hpp': Language.CPP,
        'cu': Language.CPP,
        'cuh': Language.CPP,
        'java': Language.JAVA,
        'go': Language.GO,
        'rs': Language.RUST,
    }
    
    lang = lang_map.get(ext)
    
    if lang:
        splitter = RecursiveCharacterTextSplitter.from_language(
            language=lang, 
            chunk_size=1500, 
            chunk_overlap=150
        )
        blocks = splitter.split_text(content)
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=150
        )
        blocks = splitter.split_text(content)
        
    formatted_blocks = []
    for i, b in enumerate(blocks):
        if len(b.strip()) > 20:
            formatted_blocks.append({
                "id": f"{os.path.basename(file_path)}_block{i+1}",
                "content": b.strip()
            })
            
    return formatted_blocks

In [76]:
def parse_all_raw_data(base_path, subdirs):
    all_data = {}
    for subdir in subdirs:
        folder_path = os.path.join(base_path, "data", "raw", subdir)
        if not os.path.exists(folder_path): 
            continue
        
        img_out_dir = os.path.join(base_path, "data", "images", subdir)
        
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if not os.path.isfile(file_path): 
                continue
            if filename.endswith(".mp3") or filename.endswith(".wav") or filename.endswith(".mp4"):
                continue
                
            elif filename.endswith(".pdf"):
                slides = parse_pdf_to_slides_with_images(file_path, img_out_dir)
                for slide in slides:
                    all_data[f"{subdir}_{slide['id']}"] = {
                        "type": "slide",
                        "content": slide['content'],
                        "image_path": slide['image_path']
                    }
                    
            else:
                blocks = parse_code_to_blocks(file_path)
                for block in blocks:
                    all_data[f"{subdir}_{block['id']}"] = {
                        "type": "code",
                        "content": block['content'],
                        "image_path": None
                    }
                    
    return all_data

In [77]:
class LectureDataset:
    def __init__(self, materials_dict):
        self.materials = materials_dict

    def get_by_id(self, material_id):
        return self.materials.get(material_id)

    def to_pandas(self):
        return pd.DataFrame.from_dict(self.materials, orient='index')

In [78]:
USE_MISTRAL = True

if USE_MISTRAL:
    client = Mistral(api_key=MISTRAL_API_KEY)
    generator_func = generate_slide_description_mistral
else:
    client = InferenceClient(api_key=HF_TOKEN)
    generator_func = generate_slide_description_hf

out_dir = os.path.join(BASE_PATH, "data", "linking")
os.makedirs(out_dir, exist_ok=True)

In [79]:
subdirs = ['04', '05', '06']

out_dir = os.path.join(BASE_PATH, "data", "linking")
os.makedirs(out_dir, exist_ok=True)

all_items = parse_all_raw_data(BASE_PATH, subdirs)
print(f"Found {len(all_items)} chunks of content")

for subdir in subdirs:
    synthetic_data_lec = []
    
    lec_items = {k: v for k, v in all_items.items() if k.startswith(f"{subdir}_")}
    
    if not lec_items:
        print(f"No notes found in {subdir}")
        continue

    for item_id, item_data in tqdm(lec_items.items(), desc=f"Lecture {subdir}"):
        
        content = item_data["content"]
        image_path = item_data["image_path"]
        item_type = item_data["type"]
        
        if item_type == "slide" and image_path is not None:
            variations = generator_func(client, text_content=content, image_path=image_path, num_samples=1, max_retries=1)
        else:
            variations = generate_code_description(client, text_content=content)
            
        for v in variations:
            synthetic_data_lec.append({
                "spoken_text": v,
                "context_content": content,
                "context_id": item_id,
                "image_path": image_path, 
                "source": "synthetic"
            })
        time.sleep(1)
        
    if synthetic_data_lec:
        csv_path_syn = os.path.join(out_dir, f"synthetic_{subdir}.csv")
        df_syn = pd.DataFrame(synthetic_data_lec)
        df_syn.to_csv(csv_path_syn, index=False)
        print(f"Completed {csv_path_syn}: {len(df_syn)} lines")

Found 273 chunks of content


Lecture 04:   0%|          | 0/92 [00:00<?, ?it/s]

API Error (Code): API error occurred: Status 429. Body: {"object":"error","message":"Service tier capacity exceeded for this model.","type":"service_tier_capacity_exceeded","param":null,"code":"3505","raw_status_code":429}


Lecture 04:   0%|          | 0/92 [00:09<?, ?it/s]


KeyboardInterrupt: 